# Seed check of the final configuration (Colab T4)

The submitted adapter was trained once (seed 0): clean data, 16-bit LoRA r=16, 5 epochs, lr 2e-4. This notebook trains the **same configuration with seeds 1 and 2** and scores them on the 200 dev tickets with the assignment's `score.py`, so the report can state a spread instead of a single number. It does **not** change the submitted adapter or `predictions_test.jsonl`, and it never looks at test inputs.

Run: Runtime > Change runtime type > T4 GPU, then *Run all* (about 1 hour). Results are appended to `MyDrive/nimbus/seeds_out/results.jsonl`.

In [3]:
# 0. Environment: the same library versions as the final run of notebook.ipynb (2026-11-12) ---------------------------
%pip install -q transformers==5.22.2 peft==0.21.2 accelerate bitsandbytes==0.50.2
%pip uninstall -y -q torchao

In [4]:
import os, sys, json, math, torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0), "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
assert "T4" in torch.cuda.get_device_name(0), "this check is meant for a Colab T4"
from google.colab import drive
drive.mount("/content/drive")
PACK = "/content/drive/MyDrive/nimbus/candidate_pack"      # contains data/ and score.py
OUT = "/content/drive/MyDrive/nimbus/seeds_out"            # results.jsonl, dev predictions, adapters of the seed runs
assert os.path.exists(f"{PACK}/data/train.jsonl") and os.path.exists(f"{PACK}/score.py"), "candidate_pack not found on Drive"
for d in ("src", "audit", "college"):
    os.makedirs(f"/content/seeds_code/{d}", exist_ok=True)
os.makedirs(OUT, exist_ok=True)

torch 2.14.1+cu130 | transformers 5.22.2 | peft 0.21.2
GPU: Tesla T4 | memory GB: 14.6
Mounted at /content/drive


In [5]:
%%writefile /content/seeds_code/src/nimbus_train.py
"""Training / generation helpers for the NimbusPay triage fine-tune (Qwen2.5-1.5B-Instruct + LoRA / QLoRA).

Why a hand-written loop instead of Trainer/TRL
  * API-stable across library versions (Colab upgrades them under us),
  * the loss is computed ONLY on the assistant tokens and ONLY at those positions, so the 152k-vocab logits are never
    materialised for prompt tokens (a 2,200-token quoted-thread ticket would otherwise need ~1.3 GB of fp32 logits),
  * every line is explainable in the live defense.
"""
import gc, json, math, random, re, time
import torch
import torch.nn.functional as F

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


# --------------------------------------------------------------------------- data -> tensors
def build_example(row, tok):
    """Chat-template the system+user messages, append the assistant JSON + <|im_end|>; loss only on the completion.

    The prompt part is exactly what generate() receives at test time (add_generation_prompt=True), so training and
    inference see identical prefixes. Prompt and completion are tokenised separately; a check in the notebook shows
    this equals joint tokenisation on every row.
    """
    prompt = tok.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    c = tok(row["messages"][2]["content"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"id": row["id"], "input_ids": p + c, "labels": [-100] * len(p) + c, "n_prompt": len(p), "n_target": len(c)}


def collate(exs, pad_id, device):
    L = max(len(e["input_ids"]) for e in exs)
    ids = torch.full((len(exs), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(exs), L), dtype=torch.long)
    lab = torch.full((len(exs), L), -100, dtype=torch.long)
    for i, e in enumerate(exs):                      # right padding for training
        n = len(e["input_ids"])
        ids[i, :n] = torch.tensor(e["input_ids"])
        att[i, :n] = 1
        lab[i, :n] = torch.tensor(e["labels"])
    return {"input_ids": ids.to(device), "attention_mask": att.to(device), "labels": lab.to(device)}


def micro_batches(exs, budget):
    """Split one optimiser batch into micro-batches of similar length with <= `budget` padded tokens each."""
    exs = sorted(exs, key=lambda e: len(e["input_ids"]))
    out, cur = [], []
    for e in exs:
        if cur and (len(cur) + 1) * len(e["input_ids"]) > budget:
            out.append(cur)
            cur = []
        cur.append(e)
    if cur:
        out.append(cur)
    return out


def causal_lm(model):
    """The underlying *ForCausalLM: unwraps a PeftModel (LoRA layers live inside it); a plain HF model is returned as is."""
    try:
        from peft import PeftModel
    except ImportError:
        return model
    return model.base_model.model if isinstance(model, PeftModel) else model


def loss_sum(model, batch):
    """Sum of token cross-entropy over the assistant tokens (labels != -100) and how many tokens that is."""
    inner = causal_lm(model)
    h = inner.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state
    tgt = batch["labels"][:, 1:]                                                  # position t predicts token t+1
    keep = tgt != -100
    logits = inner.lm_head(h[:, :-1][keep]).float()                               # only the completion positions
    return F.cross_entropy(logits, tgt[keep], reduction="sum"), int(keep.sum())


# --------------------------------------------------------------------------- model
def _dtype_kw(transformers, dtype):
    ver = tuple(int(x) for x in transformers.__version__.split(".")[:2])
    return {"dtype" if ver >= (4, 56) else "torch_dtype": dtype}


def load_base(model_id=MODEL_ID, quant4=False):
    import transformers
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    kw = _dtype_kw(transformers, torch.float16)
    kw.update(attn_implementation="sdpa", device_map={"": 0})
    if quant4:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(model_id, **kw)


def add_lora(base, r=16, alpha=None, dropout=0.05, quant4=False, targets=LORA_TARGETS):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    gc_kw = {"use_reentrant": False}
    if quant4:
        base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True, gradient_checkpointing_kwargs=gc_kw)
    else:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gc_kw)
        base.enable_input_require_grads()
    base.config.use_cache = False
    cfg = LoraConfig(r=r, lora_alpha=alpha or 2 * r, lora_dropout=dropout, target_modules=targets, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(base, cfg)
    for p in model.parameters():                       # fp32 adapters + fp16 autocast = stable mixed precision on a T4
        if p.requires_grad:
            p.data = p.data.float()
    return model


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    return tr, tot


def prepare_for_generation(model):
    inner = causal_lm(model)
    inner.config.use_cache = True
    model.eval()
    return model


# --------------------------------------------------------------------------- training
def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def train(model, tok, examples, *, epochs=3, lr=2e-4, batch_size=16, warmup_ratio=0.05, micro_budget=4608,
          weight_decay=0.0, max_grad_norm=1.0, seed=0, amp=True, log_every=10, verbose=True):
    """AdamW + cosine schedule with warmup. One optimiser step = `batch_size` sequences, token-mean loss."""
    from transformers import get_cosine_schedule_with_warmup
    dev = next(model.parameters()).device
    use_amp = bool(amp and dev.type == "cuda")
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    steps_per_epoch = math.ceil(len(examples) / batch_size)
    total = steps_per_epoch * epochs
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(warmup_ratio * total)), total)
    scaler = _grad_scaler(use_amp)
    rng = random.Random(seed)
    model.train()
    hist, skipped, step, t0 = [], 0, 0, time.time()
    for ep in range(epochs):
        order = list(range(len(examples)))
        rng.shuffle(order)
        run = []
        for b in range(0, len(order), batch_size):
            batch = [examples[i] for i in order[b:b + batch_size]]
            n_tgt = sum(e["n_target"] for e in batch)
            tot_loss = 0.0
            for mb in micro_batches(batch, micro_budget):
                with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                    ls, _ = loss_sum(model, collate(mb, tok.pad_token_id, dev))
                scaler.scale(ls / n_tgt).backward()           # token-mean over the whole optimiser batch
                tot_loss += ls.item()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(params, max_grad_norm)
            before = scaler.get_scale() if use_amp else None
            scaler.step(opt)
            scaler.update()
            if use_amp and scaler.get_scale() < before:
                skipped += 1                                  # fp16 overflow -> step skipped by the scaler
            opt.zero_grad(set_to_none=True)
            sched.step()
            step += 1
            loss = tot_loss / n_tgt
            run.append(loss)
            if not math.isfinite(loss) and verbose:
                print(f"  !! non-finite loss at step {step}")
            if verbose and (step % log_every == 0 or step == total):
                print(f"  ep {ep + 1}/{epochs} step {step:4d}/{total} loss {sum(run[-log_every:]) / len(run[-log_every:]):.4f} "
                      f"lr {sched.get_last_lr()[0]:.2e} |g| {float(gn):.2f}  {(time.time() - t0) / 60:.1f} min")
            hist.append((step, loss))
    return {"loss_hist": hist, "seconds": time.time() - t0, "steps": step, "skipped_steps": skipped}


# --------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, tok, rows, *, batch_size=16, max_new_tokens=200, system_override=None, **gen_kwargs):
    """Greedy decoding exactly as in the assignment: chat template on [system, user], do_sample=False, 200 new tokens.

    Nothing else is changed: the model's own generation_config (incl. its repetition_penalty) stays in force unless the
    caller passes an override, which is how the sensitivity check in the notebook works.
    Returns the raw decoded text with only .strip() applied.
    """
    prepare_for_generation(model)
    dev = next(model.parameters()).device
    old_side = tok.padding_side
    tok.padding_side = "left"
    prompts = []
    for r in rows:
        msgs = r["messages"][:2]
        if system_override is not None:
            msgs = [{"role": "system", "content": system_override}, msgs[1]]
        prompts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    order = sorted(range(len(rows)), key=lambda i: -len(prompts[i]))      # similar lengths together -> little padding
    outs = [None] * len(rows)
    for b in range(0, len(order), batch_size):
        idx = order[b:b + batch_size]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(dev)
        out = model.generate(**enc, do_sample=False, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **gen_kwargs)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for i, t in zip(idx, texts):
            outs[i] = t.strip()
    tok.padding_side = old_side
    return outs


def lenient_json(text):
    """Diagnostic only (never used for the official score): strip code fences / prose around the first {...}."""
    if text is None:
        return None
    t = re.sub(r"```(?:json)?", "", text).strip()
    m = re.search(r"\{.*\}", t, re.S)
    return m.group(0) if m else t


# --------------------------------------------------------------------------- bookkeeping
def peak_gb():
    return torch.cuda.max_memory_allocated() / 2 ** 30 if torch.cuda.is_available() else float("nan")


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Writing /content/seeds_code/src/nimbus_train.py


In [6]:
%%writefile /content/seeds_code/audit/nimbus_lib.py
"""Shared helpers for the audit: parse a ticket and derive each label field from SCHEMA.md rules.

Everything here uses only SCHEMA.md and the ticket text. No dev/test labels are used.
"""
import json, re, datetime as dt
from decimal import Decimal

ROOT = "E:/Fine_Tuneing-Assigh/training_assessment/candidate_pack/data/"
KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
NULL_CATS = {"account_access", "kyc", "other"}  # transaction fields must be null for these


def load(name):
    import os
    path = name if os.path.isabs(name) else ROOT + name
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


# ---------------------------------------------------------------- ticket parsing
HDR = re.compile(r"^Received: (\d{4})-(\d{2})-(\d{2}) \((\w{3})\)\nSource: ([\w-]+)\n\n", re.S)


def parse_ticket(user):
    m = HDR.match(user)
    received = dt.date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
    body = user[m.end():]
    own = "\n".join(l for l in body.split("\n") if not l.lstrip().startswith(">"))  # drop quoted earlier emails
    return received, m.group(5), body, own


# ---------------------------------------------------------------- amounts
NUM = r"\d{1,3}(?:,\d{2,3})+(?:\.\d+)?|\d+(?:\.\d+)?"
UNIT = r"k|lakhs?|lacs?"
AMT = re.compile(
    rf"(?P<pre>\u20b9|\bRs\.?|\bINR)\s*(?P<n1>{NUM})(?:\s*(?P<u1>{UNIT})\b)?"
    rf"|(?<![A-Za-z0-9.,])(?P<n2>{NUM})\s*(?P<u2>{UNIT})\b"
    rf"|(?<![A-Za-z0-9.,])(?P<n3>{NUM})\s*(?:rupees|rupaye|rupaiye)\b",
    re.I,
)
DISTRACT_LEFT = re.compile(r"balance|daily limit|limit|\bfees?\b|charges?\b|\bmein abhi\b|\babhi\b", re.I)
DISTRACT_RIGHT = re.compile(r"^\s*(?:right now|pade hain|baki|bacha)", re.I)


def to_paise(num, unit):
    v = Decimal(num.replace(",", ""))
    u = (unit or "").lower()
    if u == "k":
        v *= 1000
    elif u.startswith("lakh") or u.startswith("lac"):
        v *= 100000
    return int(v * 100)


def amounts(text):
    """All currency-like amounts in text -> list of dict(paise, raw, distractor)."""
    out = []
    for m in AMT.finditer(text):
        num = m.group("n1") or m.group("n2") or m.group("n3")
        unit = m.group("u1") or m.group("u2")
        left = text[: m.start()]
        cut = max(left.rfind(". "), left.rfind("! "), left.rfind("? "), left.rfind("\n"))
        sent_left = left[cut + 1:]
        dis = bool(DISTRACT_LEFT.search(sent_left)) or bool(DISTRACT_RIGHT.match(text[m.end():]))
        out.append({"paise": to_paise(num, unit), "raw": m.group(0), "distractor": dis})
    return out


# ---------------------------------------------------------------- dates
MON = {m: i for i, m in enumerate(["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}
MONRE = r"(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|June?|July?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
ABS_NUM = re.compile(r"(?<![\d/-])(\d{1,2})[/-](\d{1,2})[/-](\d{4}|\d{2})(?![\d/-])")
ABS_DM = re.compile(rf"(?<!\d)(\d{{1,2}})(?:st|nd|rd|th)?\s+{MONRE}\b(?:,?\s+(\d{{4}}))?", re.I)
ABS_MD = re.compile(
    rf"\b{MONRE}\.?\s+(\d{{1,2}})(?:st|nd|rd|th)?\b(?!\s*:)(?!\.\d)(?!\s*(?:k\b|lakh|lac\b|rupe))(?:,?\s+(\d{{4}}))?", re.I
)  # "Aug 14" / "August 14th 2026"; not "Aug. 1.3k" (a sentence end followed by an amount)
REL = [
    (re.compile(r"day before yesterday|\bparso\b", re.I), -2),
    (re.compile(r"(?<!day before )\byesterday\b|\bkal\b", re.I), -1),
    (re.compile(r"\btoday\b|\baaj\b", re.I), 0),
]
REL_N = re.compile(r"\b(\d+)\s+(?:days?\s+ago|din\s+pehle)", re.I)


def _safe(y, m, d):
    try:
        return dt.date(y, m, d)
    except ValueError:
        return None


def _no_year(received, m, d):
    c = _safe(received.year, m, d)
    if c is None:
        return None
    return c if c <= received else _safe(received.year - 1, m, d)


def dates(text, received):
    """Candidate transaction dates mentioned in text -> list of (date, kind, raw)."""
    out = []
    for m in ABS_NUM.finditer(text):
        d, mo, y = int(m.group(1)), int(m.group(2)), m.group(3)
        y = int(y) + 2000 if len(y) == 2 else int(y)
        dd = _safe(y, mo, d)
        if dd:
            out.append((dd, "abs_num", m.group(0)))
    for m in ABS_DM.finditer(text):
        d, mo, y = int(m.group(1)), MON[m.group(2)[:3].lower()], m.group(3)
        dd = _safe(int(y), mo, d) if y else _no_year(received, mo, d)
        if dd:
            out.append((dd, "abs_dm", m.group(0)))
    for m in ABS_MD.finditer(text):
        mo, d, y = MON[m.group(1)[:3].lower()], int(m.group(2)), m.group(3)
        dd = _safe(int(y), mo, d) if y else _no_year(received, mo, d)
        if dd:
            out.append((dd, "abs_md", m.group(0)))
    for rx, off in REL:
        for m in rx.finditer(text):
            out.append((received + dt.timedelta(days=off), "rel", m.group(0)))
    for m in REL_N.finditer(text):
        out.append((received - dt.timedelta(days=int(m.group(1))), "rel", m.group(0)))
    return out


# ---------------------------------------------------------------- txn id
ID_RX = re.compile(r"(?<![A-Za-z])N\s*P(?:[\s-]*\d){10}(?!\d)", re.I)


def txn_ids(text):
    return ["NP" + re.sub(r"\D", "", m.group(0)) for m in ID_RX.finditer(text)]


# ---------------------------------------------------------------- channel
CH_STRICT = [
    (re.compile(r"Mode:\s*(upi|net ?banking|internet banking|wallet|credit card|debit card|card)\b", re.I), 1),
    (re.compile(r"It was an? (upi|net ?banking|internet banking|wallet|credit card|debit card|card) transaction", re.I), 1),
    (re.compile(r"\b(upi|net ?banking|wallet|card|credit card|debit card) se hua tha", re.I), 1),
    (re.compile(r"\b(credit card|debit card|card) wala transaction tha", re.I), 1),
    (re.compile(r"(from the NimbusPay wallet)", re.I), 1),
]
CH_LOOSE = [
    ("upi", re.compile(r"\bUPI\b", re.I)),
    ("netbanking", re.compile(r"net\s?banking|internet banking", re.I)),
    ("wallet", re.compile(r"\bwallet\b", re.I)),
    ("card", re.compile(r"credit card|debit card|card payment|card transaction|\bvisa\b|mastercard|\bcard\b", re.I)),
]


def _canon_ch(s):
    s = s.lower()
    if "upi" in s:
        return "upi"
    if "bank" in s:
        return "netbanking"
    if "wallet" in s:
        return "wallet"
    return "card"


def channels_strict(text):
    return sorted({_canon_ch(m.group(1)) for rx, _ in CH_STRICT for m in rx.finditer(text)})


def channels_loose(text):
    return sorted({n for n, rx in CH_LOOSE if rx.search(text)})


# ---------------------------------------------------------------- language, contact, legal
HI_WORDS = set("""hai hain hoon tha thi the kya kaise kyun kyu nahi nahin mera meri mere mujhe aap apna apni ko ka ki ke se par lekin
paise paisa wapas kat kaat gaye gaya hua hui ho kar karo karna karun kiya raha rahi rahe abhi tak jaldi dhanyavaad namaste bhai ji baar
teesri pehle pichla din rakam kitna kinta chahiye jaunga dunga bol bata batao dikha dikh mein kuch koi aur bhi hi toh to ek do
saamaan ghar phir fir chuka chala chali samajh hua hoga kijiye kijie kare kar sakta sakte""".split())
HI_STRONG = set("""hai hain hoon kya kaise nahi nahin mera meri mere mujhe lekin paise paisa wapas kat kaat gaye gaya raha rahi abhi jaldi
dhanyavaad namaste baar teesri pehle rakam kitna chahiye jaunga dunga bol bata karo karna karun kiya hua tha hoon""".split())


def hindi_score(text):
    toks = re.findall(r"[a-z]+", text.lower())
    return sum(t in HI_STRONG for t in toks), len(toks)


CONTACT = re.compile(
    r"third time|teesri baar|already written|contacted support|second complaint|again and again|baar baar|pehle bhi complaint",
    re.I,
)
TICKET_REF = re.compile(r"earlier ticket number|pichla ticket number", re.I)
LEGAL = re.compile(r"\bRBI\b|ombudsman|consumer court|lawyer|legal notice|legal action", re.I)


# ---------------------------------------------------------------- rules from SCHEMA.md
def exp_priority(cat, amount, legal):
    if cat == "fraud" or (amount is not None and amount >= 5_000_000) or legal:
        return "P1"
    if (cat in ("payment_failed", "refund") and amount is not None and amount >= 500_000) or cat == "account_access":
        return "P2"
    return "P3"


def exp_needs_human(priority, contacted):
    return priority == "P1" or contacted


def parse_label(content):
    try:
        lab = json.loads(content)
    except Exception:
        return None
    return lab if isinstance(lab, dict) else None


# ---------------------------------------------------------------- full derivation (spec + text -> expected label fields)
SCRATCH = re.compile(r"scratch\s*card|srcatch\s*card", re.I)


def channel_from_text(own):
    st = channels_strict(own)
    if len(st) == 1:
        return st[0]
    if len(st) > 1:
        return None
    lo = channels_loose(SCRATCH.sub(" ", own))
    return lo[0] if len(lo) == 1 else None


def derive(user, category):
    """Expected values for every deterministic field, given the (trusted) category.

    Returns dict with the fields plus 'ambiguous' = list of fields where the text gave >1 candidate.
    """
    rec, src, body, own = parse_ticket(user)
    amb = []
    if category in NULL_CATS:
        amount = date = tid = channel = None
    else:
        main = sorted({a["paise"] for a in amounts(own) if not a["distractor"]})
        if len(main) > 1:
            amb.append("amount_paise")
        amount = main[0] if len(main) == 1 else None
        cands = sorted({d.isoformat() for d, _, _ in dates(own, rec)})
        if len(cands) > 1:
            amb.append("txn_date")
        date = cands[0] if len(cands) == 1 else None
        ids = sorted(set(txn_ids(own)))
        if len(ids) > 1:
            amb.append("txn_id")
        tid = ids[0] if len(ids) == 1 else None
        channel = channel_from_text(own)
    legal, contacted = bool(LEGAL.search(own)), bool(CONTACT.search(own))
    prio = exp_priority(category, amount, legal)
    hs, _ = hindi_score(own)
    return {
        "category": category, "priority": prio, "amount_paise": amount, "txn_date": date, "txn_id": tid,
        "channel": channel, "language": "hi-en" if hs >= 2 else "en",
        "needs_human": exp_needs_human(prio, contacted), "ambiguous": amb,
        "_legal": legal, "_contacted": contacted,
    }

Writing /content/seeds_code/audit/nimbus_lib.py


In [7]:
%%writefile /content/seeds_code/audit/nimbus_clean.py
"""Cleaning of train.jsonl (Part A). Uses only train.jsonl + SCHEMA.md rules (nimbus_lib.derive).

One exception, disclosed in the report: dev *inputs* (never dev labels) are used to find train rows that are copies of
dev tickets, because training on them would break the "never train on dev" rule.
"""
import json, re, collections
from nimbus_lib import *

JUNK = re.compile(r"^\s*(?:test|asdf asdf|test test \d+)?\s*$", re.I)
LEGACY_PRIO = {"low": "P3", "medium": "P2", "high": "P1"}


def norm_id(s):
    return "NP" + re.sub(r"\D", "", s) if s else s


def clean_row(r):
    """Return (new_label_dict, [fix reasons]) or (None, [drop reasons])."""
    user = r["messages"][1]["content"]
    lab = parse_label(r["messages"][2]["content"])
    fixes = []

    # ---- v1 (legacy) format: prio low/medium/high + dd/mm/yyyy dates
    if "prio" in lab:
        old = lab["prio"]
        lab = {("priority" if k == "prio" else k): v for k, v in lab.items()}
        lab["priority"] = LEGACY_PRIO[old]
        if lab["txn_date"]:
            dd, mm, yy = lab["txn_date"].split("/")
            lab["txn_date"] = f"{yy}-{mm}-{dd}"
        fixes.append(f"legacy v1 format: 'prio':'{old}'->'priority':'{lab['priority']}', date dd/mm/yyyy->ISO")
    lab = {k: lab[k] for k in KEYS}   # canonical key order
    d = derive(user, lab["category"])
    if d["ambiguous"]:
        return None, [f"ambiguous text for {d['ambiguous']}; label cannot be verified"]

    # ---- amount: rupees stored instead of paise
    if lab["amount_paise"] != d["amount_paise"]:
        t = d["amount_paise"]
        if lab["amount_paise"] is not None and t is not None and lab["amount_paise"] == t // 100:
            fixes.append(f"amount_paise stored in rupees ({lab['amount_paise']}) instead of paise ({t})")
            lab["amount_paise"] = t
        else:
            return None, [f"amount_paise {lab['amount_paise']} unexplained (text gives {t})"]
    # ---- txn_id: raw text copied instead of normalised NP+10 digits
    if lab["txn_id"] != d["txn_id"]:
        if lab["txn_id"] is not None and norm_id(lab["txn_id"]) == d["txn_id"]:
            fixes.append(f"txn_id not normalised ('{lab['txn_id']}' -> '{d['txn_id']}')")
            lab["txn_id"] = d["txn_id"]
        else:
            return None, [f"txn_id {lab['txn_id']} unexplained (text gives {d['txn_id']})"]
    # ---- remaining deterministic fields must already agree
    for k in ("txn_date", "channel", "language"):
        if lab[k] != d[k]:
            return None, [f"{k} {lab[k]!r} unexplained (text/spec gives {d[k]!r})"]
    # ---- priority: recompute from spec. Evidence that the *priority* column is the corrupted one:
    #      needs_human always agrees with the spec-derived priority, never with the bad label.
    if lab["needs_human"] != d["needs_human"]:
        return None, [f"needs_human {lab['needs_human']} unexplained (spec gives {d['needs_human']})"]
    if lab["priority"] != d["priority"]:
        fixes.append(
            f"priority {lab['priority']} contradicts spec -> {d['priority']} "
            f"(category={lab['category']}, amount_paise={lab['amount_paise']}, legal_threat={d['_legal']}); needs_human agrees with {d['priority']}"
        )
        lab["priority"] = d["priority"]
    return lab, fixes


def clean_dataset(rows, dev_rows):
    """-> (kept_rows, log[(id, action, reason)], stats Counter)"""
    dev_inputs = {r["messages"][1]["content"]: r["id"] for r in dev_rows}
    log, kept, seen_full = [], [], {}
    stats = collections.Counter()
    for r in rows:
        rid, user = r["id"], r["messages"][1]["content"]
        own = parse_ticket(user)[3] if HDR.match(user) else ""
        if user in dev_inputs:                                   # 1. leakage of dev tickets into train
            log.append((rid, "drop", f"leakage: ticket text identical to dev ticket {dev_inputs[user]} (never train on dev)"))
            stats["drop_dev_leak"] += 1
            continue
        if JUNK.match(own):                                      # 2. junk / empty ticket with an unsupported label
            log.append((rid, "drop", f"junk ticket (body={own.strip()[:15]!r}): label has fields the text cannot support"))
            stats["drop_junk"] += 1
            continue
        if parse_label(r["messages"][2]["content"]) is None:     # 3. truncated label
            log.append((rid, "drop", "assistant label is truncated / invalid JSON"))
            stats["drop_truncated"] += 1
            continue
        if user in seen_full:                                    # 4. exact duplicate of an earlier kept ticket
            log.append((rid, "drop", f"exact duplicate of {seen_full[user]} (same text and label)"))
            stats["drop_duplicate"] += 1
            continue
        lab, notes = clean_row(r)
        if lab is None:
            log.append((rid, "drop", "; ".join(notes)))
            stats["drop_unexplained"] += 1
            continue
        if notes:
            log.append((rid, "fix", "; ".join(notes)))
            for key, pat in (("fix_legacy", "legacy v1"), ("fix_amount", "stored in rupees"), ("fix_txn_id", "not normalised"), ("fix_priority", "contradicts spec")):
                if pat in notes[0]:
                    stats[key] += 1
        seen_full[user] = rid
        kept.append({"id": rid, "messages": [r["messages"][0], r["messages"][1], {"role": "assistant", "content": json.dumps(lab)}]})
    return kept, log, stats

Writing /content/seeds_code/audit/nimbus_clean.py


In [8]:
%%writefile /content/seeds_code/college/run_college.py
"""Run fine-tuning experiments on a machine that is NOT the Colab T4 (the college PC) and package the results.

Everything this script produces is NON-T4: say so in the report. Memory and time figures from here are not comparable
with the T4 table, and the final submitted adapter must still be reproduced on a fresh Colab T4.

    python run_college.py --pack ../candidate_pack --jobs R0_repeat,R0b_seed1,D3b_5epochs,D4_keyorder

Output: <out>/college_results.zip  (adapters + dev predictions + results.jsonl + environment). It contains NO company data.
"""
import argparse, collections, csv, json, os, platform, sys, time, zipfile

HERE = os.path.dirname(os.path.abspath(__file__))
for base in (HERE, os.path.dirname(HERE)):          # works from the zip layout and from the repo layout
    for sub in ("src", "audit"):
        p = os.path.join(base, sub)
        if os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)

# name -> overrides of the R0 configuration (clean data, 16-bit LoRA r=16, lr 2e-4, 3 epochs, seed 0)
JOBS = {
    "R0_repeat":    dict(desc="R0 configuration repeated on this machine (hardware check)"),
    "R0b_seed1":    dict(desc="R0 with seed 1 (seed noise)", seed=1),
    "D3a_1epoch":   dict(desc="1 epoch", epochs=1),
    "D3b_5epochs":  dict(desc="5 epochs", epochs=5),
    "D4_keyorder":  dict(desc="derived fields (priority, needs_human) emitted last", reorder=True),
    "D5_rank8":     dict(desc="LoRA rank 8", r=8),
    "D5_rank32":    dict(desc="LoRA rank 32 (36.9M params)", r=32),
    "D6_lr1e-4":    dict(desc="learning rate 1e-4", lr=1e-4),
    "D6_lr4e-4":    dict(desc="learning rate 4e-4", lr=4e-4),
    "D2_qlora4":    dict(desc="4-bit QLoRA", quant4=True),
    # FALLBACK ONLY (T4 unavailable): the final configuration (clean data, 16-bit LoRA r=16, 5 epochs) + test predictions via --test.
    "FINAL_5ep":    dict(desc="final configuration: clean data, 16-bit LoRA r=16, 5 epochs (non-T4 fallback)", epochs=5),
}
DEFAULTS = dict(quant4=False, r=16, lr=2e-4, epochs=3, dropout=0.05, seed=0, reorder=False)
ORDER = ["category", "amount_paise", "txn_date", "txn_id", "channel", "language", "priority", "needs_human"]


def reorder(rows):
    out = []
    for r in rows:
        lab = json.loads(r["messages"][2]["content"])
        out.append({"id": r["id"], "messages": [r["messages"][0], r["messages"][1],
                    {"role": "assistant", "content": json.dumps({k: lab[k] for k in ORDER})}]})
    return out


def tiny_base():
    """Smoke-test model (random weights, CPU). Only used with --smoke."""
    from transformers import Qwen2Config, Qwen2ForCausalLM
    cfg = Qwen2Config(vocab_size=151936, hidden_size=64, intermediate_size=128, num_hidden_layers=2, num_attention_heads=4,
                      num_key_value_heads=2, max_position_embeddings=4096, tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).float()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pack", required=True, help="folder with data/train.jsonl, data/dev.jsonl, score.py, SCHEMA.md")
    ap.add_argument("--out", default="college_out")
    ap.add_argument("--jobs", default="R0_repeat,R0b_seed1,D3b_5epochs,D4_keyorder", help="comma list from: " + ", ".join(JOBS))
    ap.add_argument("--model", default=None, help="HF id or a local folder (default Qwen/Qwen2.5-1.5B-Instruct)")
    ap.add_argument("--smoke", action="store_true", help="CPU smoke test with a tiny random model")
    ap.add_argument("--test", default=None, help="optional path to test_inputs.jsonl: also write test predictions (greedy, batch size 1) per job")
    ap.add_argument("--limit-train", type=int, default=0)
    ap.add_argument("--limit-dev", type=int, default=0)
    a = ap.parse_args()

    import numpy as np, torch, transformers
    import nimbus_lib as nl, nimbus_clean as nc, nimbus_train as nt
    from transformers import AutoTokenizer
    try:
        import peft
        peft_v = peft.__version__
    except ImportError:
        sys.exit("peft is not installed")
    sys.path.insert(0, a.pack)
    import score as scorer

    cuda = torch.cuda.is_available()
    if not cuda and not a.smoke:
        sys.exit("No CUDA GPU found. Use --smoke only for a CPU test.")
    gpu = torch.cuda.get_device_name(0) if cuda else "cpu"
    env = {"hardware": gpu, "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if cuda else None,
           "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft_v, "python": platform.python_version(),
           "platform": platform.platform(), "note": "NON-T4 run: label as such in the report"}
    print("environment:", env)
    if cuda and "T4" not in gpu:
        print(f"NOTE: {gpu} is not a T4. Dev scores are comparable; memory/time are not.")

    out = os.path.abspath(a.out)
    for d in ("adapters", "preds", "logs"):
        os.makedirs(os.path.join(out, d), exist_ok=True)

    # ---- data: same cleaning as the notebook (Part A)
    nl.ROOT = os.path.join(a.pack, "data") + os.sep
    raw, dev = nl.load("train.jsonl"), nl.load("dev.jsonl")
    clean, log, stats = nc.clean_dataset(raw, dev)
    print(f"cleaning: {len(raw)} -> {len(clean)} rows  {dict(stats)}")
    assert stats["drop_unexplained"] == 0
    if not a.smoke:
        assert len(clean) == 1320, "unexpected cleaned size: is this the original train.jsonl?"
    raw_texts = {r["messages"][1]["content"] for r in raw}
    leak_ids = {r["id"] for r in dev if r["messages"][1]["content"] in raw_texts}
    train_rows = clean[: a.limit_train] if a.limit_train else clean
    dev_rows = dev[: a.limit_dev] if a.limit_dev else dev

    model_id = a.model or nt.MODEL_ID
    tok = AutoTokenizer.from_pretrained(model_id)

    def evaluate(texts):
        preds = [{"id": g["id"], "output": t} for g, t in zip(dev_rows, texts)]
        res = scorer.score(dev_rows, preds)["all"]
        keep = [g for g in dev_rows if g["id"] not in leak_ids]
        res_c = scorer.score(keep, preds)["all"] if keep else res
        return {"exact": res["exact_match"], "field_acc": res["mean_field_acc"], "json_valid": res["json_valid"],
                "exact_clean_dev": res_c["exact_match"], "fields": res["fields"]}, preds

    results_path = os.path.join(out, "results.jsonl")
    for name in [j.strip() for j in a.jobs.split(",") if j.strip()]:
        if name not in JOBS:
            print(f"skipping unknown job {name!r}; choose from {list(JOBS)}")
            continue
        cfg = {**DEFAULTS, **{k: v for k, v in JOBS[name].items() if k != "desc"}}
        print(f"\n=== {name}: {JOBS[name]['desc']}  cfg={cfg}")
        try:
            nt.free_memory()
            if cuda:
                assert torch.cuda.memory_allocated() < 0.5 * 2**30, "GPU memory not free at job start"
                torch.cuda.reset_peak_memory_stats()
            nt.seed_everything(cfg["seed"])
            base = tiny_base() if a.smoke else nt.load_base(model_id, quant4=cfg["quant4"])
            model = nt.add_lora(base, r=cfg["r"], dropout=cfg["dropout"], quant4=cfg["quant4"])
            trainable, total = nt.count_params(model)
            assert trainable <= 50e6, f"{trainable / 1e6:.1f}M trainable parameters exceeds the 50M cap"
            rows = reorder(train_rows) if cfg["reorder"] else train_rows
            exs = [nt.build_example(x, tok) for x in rows]
            print(f"  rows {len(exs)} | trainable {trainable / 1e6:.2f}M of {total / 1e6:.0f}M")
            tr = nt.train(model, tok, exs, epochs=1 if a.smoke else cfg["epochs"], lr=cfg["lr"], batch_size=16, seed=cfg["seed"])
            train_min, peak = tr["seconds"] / 60, nt.peak_gb()
            model.save_pretrained(os.path.join(out, "adapters", name))
            t0 = time.time()
            texts = nt.generate(model, tok, dev_rows, batch_size=16, max_new_tokens=20 if a.smoke else 200)
            ev, preds = evaluate(texts)
            texts2 = nt.generate(model, tok, dev_rows, batch_size=16, max_new_tokens=20 if a.smoke else 200, repetition_penalty=1.0)
            ev2, _ = evaluate(texts2)
            entry = {"name": name, "desc": JOBS[name]["desc"], "hardware": gpu, "cfg": cfg, "n_train": len(exs), "trainable_M": trainable / 1e6,
                     **ev, "exact_rep1.0": ev2["exact"], "field_acc_rep1.0": ev2["field_acc"], "peak_gb": peak, "train_min": train_min,
                     "eval_min": (time.time() - t0) / 60, "skipped_steps": tr["skipped_steps"],
                     "final_loss": float(np.mean([l for _, l in tr["loss_hist"][-10:]]))}
            with open(results_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(entry) + "\n")
            with open(os.path.join(out, "preds", f"{name}_dev.jsonl"), "w", encoding="utf-8") as f:
                for p in preds:
                    f.write(json.dumps(p, ensure_ascii=False) + "\n")
            if a.test:                                          # test predictions: same call as the notebook (plain greedy, batch size 1)
                test_rows = [json.loads(l) for l in open(a.test, encoding="utf-8") if l.strip()]
                if a.limit_dev:
                    test_rows = test_rows[: a.limit_dev]
                t1 = time.time()
                test_texts = nt.generate(model, tok, test_rows, batch_size=1, max_new_tokens=20 if a.smoke else 200)
                n_valid = 0
                with open(os.path.join(out, "preds", f"{name}_test.jsonl"), "w", encoding="utf-8") as f:
                    for r, o in zip(test_rows, test_texts):
                        f.write(json.dumps({"id": r["id"], "output": o}, ensure_ascii=False) + "\n")
                        try:
                            n_valid += int(isinstance(json.loads(o), dict) and len(json.loads(o)) == 8)
                        except ValueError:
                            pass
                entry["test_rows"], entry["test_valid_8key"], entry["test_min"] = len(test_rows), n_valid, (time.time() - t1) / 60
                with open(results_path, "a", encoding="utf-8") as f:
                    f.write(json.dumps({"name": name + "__test", "hardware": gpu, "test_rows": len(test_rows),
                                        "test_valid_8key": n_valid, "test_min": entry["test_min"]}) + "\n")
                print(f"  -> test predictions: {n_valid}/{len(test_rows)} valid 8-key JSON in {entry['test_min']:.1f} min")
            with open(os.path.join(out, "logs", f"{name}_loss.json"), "w") as f:
                json.dump(tr["loss_hist"], f)
            print(f"  -> exact {ev['exact']:.1%} | field {ev['field_acc']:.1%} | valid {ev['json_valid']:.1%} | clean-dev exact "
                  f"{ev['exact_clean_dev']:.1%} | rep_penalty=1.0 exact {ev2['exact']:.1%} | peak {peak:.1f} GB | train {train_min:.1f} min")
            del model, base
        except Exception as e:                                  # one failing job must not waste the others
            import traceback
            traceback.print_exc()
            with open(results_path, "a", encoding="utf-8") as f:
                f.write(json.dumps({"name": name, "hardware": gpu, "error": repr(e)}) + "\n")
        finally:
            nt.free_memory()

    # ---- summary + zip (no company data inside)
    rows = [json.loads(l) for l in open(results_path, encoding="utf-8")] if os.path.exists(results_path) else []
    with open(os.path.join(out, "summary.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["name", "hardware", "exact", "field_acc", "json_valid", "exact_clean_dev", "exact_rep1.0", "peak_gb", "train_min", "error"])
        for r in rows:
            w.writerow([r.get(k, "") for k in ("name", "hardware", "exact", "field_acc", "json_valid", "exact_clean_dev", "exact_rep1.0", "peak_gb", "train_min", "error")])
    with open(os.path.join(out, "environment.json"), "w") as f:
        json.dump(env, f, indent=1)
    zpath = os.path.join(out, "college_results.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for folder, _, files in os.walk(out):
            for fn in files:
                full = os.path.join(folder, fn)
                if full != zpath:
                    z.write(full, os.path.relpath(full, out))
    print(f"\nDONE. Copy this file back to the laptop: {zpath}  ({os.path.getsize(zpath) / 2**20:.0f} MB)")


if __name__ == "__main__":
    main()

Writing /content/seeds_code/college/run_college.py


In [9]:
# Two extra jobs = the FINAL configuration (clean data, 16-bit LoRA r=16, 5 epochs) with seeds 1 and 2. Everything else is the
# runner's default (lr 2e-4, dropout 0.05, batch 16, MAX_LEN 2560), which is what the final notebook used for seed 0.
sys.path.insert(0, "/content/seeds_code/college")
import run_college as rc
rc.JOBS["FINAL_5ep_seed1"] = dict(desc="final configuration (5 epochs), seed 1", epochs=5, seed=1)
rc.JOBS["FINAL_5ep_seed2"] = dict(desc="final configuration (5 epochs), seed 2", epochs=5, seed=2)
sys.argv = ["run_college.py", "--pack", PACK, "--out", OUT, "--jobs", "FINAL_5ep_seed1,FINAL_5ep_seed2"]
rc.main()                      # about 28 minutes per seed on a T4; each finished seed is appended to results.jsonl at once

environment: {'hardware': 'Tesla T4', 'vram_gb': 14.6, 'torch': '2.14.1+cu130', 'transformers': '5.22.2', 'peft': '0.21.2', 'python': '3.13.16', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'note': 'NON-T4 run: label as such in the report'}
cleaning: 1445 -> 1320 rows  {'drop_dev_leak': 30, 'fix_priority': 60, 'fix_legacy': 70, 'drop_junk': 10, 'drop_truncated': 25, 'fix_txn_id': 20, 'fix_amount': 25, 'drop_duplicate': 60}


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  rows 1320 | trainable 18.46M of 1562M
  ep 1/5 step   10/415 loss 1.6776 lr 1.00e-04 |g| 2.10  0.5 min
  ep 1/5 step   20/415 loss 0.3001 lr 2.00e-04 |g| 0.59  1.0 min
  ep 1/5 step   30/415 loss 0.0617 lr 2.00e-04 |g| 0.38  1.6 min
  ep 1/5 step   40/415 loss 0.0382 lr 1.99e-04 |g| 0.27  2.1 min
  ep 1/5 step   50/415 loss 0.0358 lr 1.97e-04 |g| 0.15  2.6 min
  ep 1/5 step   60/415 loss 0.0330 lr 1.95e-04 |g| 0.25  3.1 min
  ep 1/5 step   70/415 loss 0.0302 lr 1.92e-04 |g| 0.09  3.7 min
  ep 1/5 step   80/415 loss 0.0229 lr 1.89e-04 |g| 0.61  4.2 min
  ep 2/5 step   90/415 loss 0.0121 lr 1.85e-04 |g| 0.23  4.8 min
  ep 2/5 step  100/415 loss 0.0141 lr 1.80e-04 |g| 0.31  5.3 min
  ep 2/5 step  110/415 loss 0.0068 lr 1.75e-04 |g| 0.21  5.9 min
  ep 2/5 step  120/415 loss 0.0061 lr 1.70e-04 |g| 0.06  6.5 min
  ep 2/5 step  130/415 loss 0.0061 lr 1.64e-04 |g| 0.11  7.1 min
  ep 2/5 step  140/415 loss 0.0040 lr 1.58e-04 |g| 0.02  7.6 min
  ep 2/5 step  150/415 loss 0.0051 lr 1.51e-04 |g|

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  rows 1320 | trainable 18.46M of 1562M
  ep 1/5 step   10/415 loss 1.6661 lr 1.00e-04 |g| 1.92  0.6 min
  ep 1/5 step   20/415 loss 0.3058 lr 2.00e-04 |g| 0.65  1.2 min
  ep 1/5 step   30/415 loss 0.0606 lr 2.00e-04 |g| 0.39  1.7 min
  ep 1/5 step   40/415 loss 0.0419 lr 1.99e-04 |g| 0.17  2.2 min
  ep 1/5 step   50/415 loss 0.0325 lr 1.97e-04 |g| 0.27  2.8 min
  ep 1/5 step   60/415 loss 0.0253 lr 1.95e-04 |g| 0.44  3.3 min
  ep 1/5 step   70/415 loss 0.0237 lr 1.92e-04 |g| 0.17  3.8 min
  ep 1/5 step   80/415 loss 0.0161 lr 1.89e-04 |g| 0.35  4.3 min
  ep 2/5 step   90/415 loss 0.0166 lr 1.85e-04 |g| 0.23  4.9 min
  ep 2/5 step  100/415 loss 0.0106 lr 1.80e-04 |g| 0.20  5.4 min
  ep 2/5 step  110/415 loss 0.0085 lr 1.75e-04 |g| 0.14  5.9 min
  ep 2/5 step  120/415 loss 0.0054 lr 1.70e-04 |g| 0.08  6.5 min
  ep 2/5 step  130/415 loss 0.0079 lr 1.64e-04 |g| 0.13  7.0 min
  ep 2/5 step  140/415 loss 0.0074 lr 1.58e-04 |g| 0.05  7.6 min
  ep 2/5 step  150/415 loss 0.0057 lr 1.51e-04 |g|

In [10]:
# Summary: seed 0 is the final notebook's run (copied from its results.jsonl), seeds 1 and 2 come from this notebook.
rows = [json.loads(l) for l in open(f"{OUT}/results.jsonl", encoding="utf-8") if l.strip()]
runs = {"seed 0 (final notebook)": dict(exact=0.94, field_acc=0.98875, exact_clean_dev=0.9352941176470588, **{"exact_rep1.0": 0.94}, train_min=24.099143981933594, peak_gb=5.379127502441406)}
for r in rows:
    if r["name"].startswith("FINAL_5ep_seed") and "exact" in r:
        runs["seed " + r["name"][-1] + " (this notebook)"] = r
print(f"{'run':26s} {'exact':>7s} {'field':>7s} {'clean-dev':>10s} {'rep 1.0':>8s} {'peak GB':>8s} {'train min':>10s}")
for k, r in runs.items():
    print(f"{k:26s} {r['exact']:7.1%} {r['field_acc']:7.1%} {r['exact_clean_dev']:10.1%} {r['exact_rep1.0']:8.1%} {r['peak_gb']:8.2f} {r['train_min']:10.1f}")
ex = [r["exact"] * 100 for r in runs.values()]
fa = [r["field_acc"] * 100 for r in runs.values()]
def sd(x): return math.sqrt(sum((v - sum(x) / len(x)) ** 2 for v in x) / (len(x) - 1)) if len(x) > 1 else float("nan")
print(f"\nexact over {len(ex)} seeds: mean {sum(ex)/len(ex):.2f}, sd {sd(ex):.2f}, min {min(ex):.1f}, max {max(ex):.1f}")
print(f"field accuracy over {len(fa)} seeds: mean {sum(fa)/len(fa):.2f}, sd {sd(fa):.2f}, min {min(fa):.2f}, max {max(fa):.2f}")
errors = [r for r in rows if "error" in r]
print("failed jobs:", [(r['name'], r['error']) for r in errors] if errors else "none")

run                          exact   field  clean-dev  rep 1.0  peak GB  train min
seed 0 (final notebook)      94.0%   98.9%      93.5%    94.0%     5.38       24.1
seed 1 (this notebook)       94.0%   98.9%      94.1%    94.0%     5.35       22.1
seed 2 (this notebook)       92.0%   98.5%      91.8%    92.0%     5.36       22.2

exact over 3 seeds: mean 93.33, sd 1.15, min 92.0, max 94.0
field accuracy over 3 seeds: mean 98.75, sd 0.22, min 98.50, max 98.88
failed jobs: none
